In [2]:
import math
from datetime import datetime, timedelta, timezone
import folium
from weather_data import get_weather, load_weather, select_gfs_run

velocity = 1
launch_altitude = 0.0           # release height (m above sea level)
ground_altitude = 0.0           # landing-area elevation (m above sea level); reaching it after burst is a landing
altitude = launch_altitude
density = 1.2
volume = (math.pi / 6) * 3.0**3   # 3 m diameter balloon at fill (m^3)
balloon_mass = 2.0              # latex balloon (envelope) mass (kg)
gravity = 9.80665               # updated with altitude in the loop
time = 0
dt = 0.1
latitude = 40.8074
longitude = -73.9630
launch_time = None              # UTC datetime of release (naive = UTC), e.g. datetime(2026, 11, 14, 15, 30); None = now
drag_coefficient = 0.8          # subcritical (low Reynolds number) drag coefficient
payload_mass = 8.0
hydrogen_density = 0.0838         # kg/m^3 at 20 C, 1 atm

# Ideal gas model for hydrogen
R_hydrogen = 4124.5        # Specific gas constant for hydrogen (J/kg/K)

#simplified wind constants (used only by the offline fallback below)
wind_east = 5.0
wind_north = 2.0

# Without these flags any weather problem stops the run with an error
allow_offline_fallback = False  # True: if GFS fails, run on the standard atmosphere and constant wind (NOT a forecast)
allow_reduced_levels = False    # True: accept GFS levels only up to 100 hPa (~16 km) if the extended levels fail

max_altitude = 35000
max_time = 18000

# Live weather (GFS via weather_data.py)
weather_update_interval = 2.0   # simulated seconds between weather lookups
print_interval = 10.0           # simulated seconds between status prints
relative_humidity = 0.0

# Drag crisis: drag coefficient drops once the flow around the balloon turns turbulent
drag_coefficient_supercritical = 0.3
reynolds_crisis_start = 2.0e5
reynolds_crisis_end = 4.0e5

# Added mass: accelerating the balloon also accelerates surrounding air (0.5 for a sphere)
added_mass_coefficient = 0.5

# Ascent rate (m/s): the balloon climbs at this constant rate until burst, the way
# flight predictors such as Stratoflights (CUSF Tawhiri) do. Enter the same value there
# to compare. None uses the vertical force balance instead, which climbs too fast at
# altitude because the lifting gas is assumed to be at ambient temperature and pressure.
ascent_rate = 5.0

# Burst and parachute descent
burst_enabled = True            # False keeps the old behavior of capping volume at max_volume
# Burst diameter (m): set this from the manufacturer's rated burst diameter for the
# actual balloon. 10.54 m is typical of a 2 kg latex envelope.
burst_diameter = 10.54
max_volume = (math.pi / 6) * burst_diameter**3   # burst volume (m^3) when burst_enabled is True
parachute_diameter = 2.1        # m; lands the 8 kg payload at about 5 m/s. Set to the actual parachute.
parachute_drag_coefficient = 1.5

# Earth model
earth_radius = 6371000.0        # mean radius for gravity (m)
wgs84_a = 6378137.0             # WGS84 semi-major axis (m)
wgs84_e2 = 6.69437999014e-3     # WGS84 first eccentricity squared


#atmosphere
def atmosphere(altitude):
    altitude = max(0, 0, altitude)
    R = 287.05
    g0 = 9.80665

    layers = [
        (0, 288.15, 101325.0, -0.0065),  # Troposphere,
        (11000, 216.65, 22632.1, 0.0),  # Tropopause
        (20000, 216.65, 5474.89, 0.001),  # Stratosphere
        (32000, 228.65, 868.02, 0.0028),  # Stratopause
        (47000, 270.65, 110.91, 0.0),  # Mesosphere
        (51000, 270.65, 66.94, -0.0028),  # Mesopause
    ]
    if altitude > 51000:
        T = 270.65 - 0.0028 * (altitude - 51000)
        p = 66.94 * math.exp(-g0 * (altitude - 51000) / (R * T))
        return T, p, p / (R * T)
    if altitude > 47000:
        T = 270.65
        p = 110.91 * math.exp(-g0 * (altitude - 47000) / (R * T))
        return T, p, p / (R * T)

    for i in range(len(layers) - 1):
        h0, T0, p0, L = layers[i]
        h1, T1, p1, _ = layers[i + 1]

        if altitude < h1:
            dAltitude = altitude - h0
            if L == 0:
                T = T0
                p = p0 * math.exp(-g0 * dAltitude / (R * T))
            else:
                T = T0 + L * dAltitude
                p = p0 * (T / T0) ** (-g0 / (L * R))
            return T, p, p / (R * T)
    raise ValueError("Atmosphere model altitude out of range.")


#gravity decreases with altitude (inverse square)
def gravity_at(altitude):
    return 9.80665 * (earth_radius / (earth_radius + altitude)) ** 2


#dynamic viscosity of air, Sutherland's law (Pa s)
def air_viscosity(temperature):
    return 1.716e-5 * (temperature / 273.15) ** 1.5 * (273.15 + 110.4) / (temperature + 110.4)


#drag coefficient vs Reynolds number, blended in log(Re) across the drag crisis
def balloon_drag_coefficient(reynolds):
    if reynolds <= reynolds_crisis_start:
        return drag_coefficient
    if reynolds >= reynolds_crisis_end:
        return drag_coefficient_supercritical
    fraction = math.log(reynolds / reynolds_crisis_start) / math.log(reynolds_crisis_end / reynolds_crisis_start)
    return drag_coefficient + fraction * (drag_coefficient_supercritical - drag_coefficient)


#frost point (K): vapor pressure from RH (Bolton, as in weather_data.py), inverted with Magnus over ice
def frost_point(temperature, relative_humidity):
    temp_c = temperature - 273.15
    vapor_pressure = max(relative_humidity, 0.01) / 100.0 * 611.2 * math.exp(17.67 * temp_c / (temp_c + 243.5))
    x = math.log(vapor_pressure / 611.2)
    return 272.62 * x / (22.46 - x) + 273.15


T0, p0, rho0 = atmosphere(0)
hydrogen_mass = hydrogen_density * volume
total_mass = payload_mass + balloon_mass + hydrogen_mass

# Weather at simulation time t is the forecast valid at launch_time + t
if launch_time is None:
    launch_time = datetime.now(timezone.utc)
elif launch_time.tzinfo is None:
    launch_time = launch_time.replace(tzinfo=timezone.utc)

# Validate inputs before loading any weather
input_errors = []
if not -90 <= latitude <= 90:
    input_errors.append(f"latitude {latitude} must be within -90..90")
if not -180 <= longitude <= 180:
    input_errors.append(f"longitude {longitude} must be within -180..180")
for name, value in [("payload_mass", payload_mass), ("balloon_mass", balloon_mass),
                    ("hydrogen_density", hydrogen_density), ("volume", volume),
                    ("burst_diameter", burst_diameter), ("parachute_diameter", parachute_diameter),
                    ("dt", dt)]:
    if not value > 0:
        input_errors.append(f"{name} must be positive (got {value})")
if ascent_rate is not None and not ascent_rate > 0:
    input_errors.append(f"ascent_rate must be positive, or None for the force balance (got {ascent_rate})")
if launch_altitude < ground_altitude:
    input_errors.append(f"launch_altitude {launch_altitude} m is below ground_altitude {ground_altitude} m")
if burst_enabled and volume >= max_volume:
    input_errors.append(f"fill volume {volume:.1f} m^3 is not below the burst volume {max_volume:.1f} m^3")
try:
    select_gfs_run(launch_time)
except ValueError as error:
    input_errors.append(str(error))
if input_errors:
    raise ValueError("Invalid simulation inputs:\n  - " + "\n  - ".join(input_errors))

# A constant-rate ascent starts at that rate
if ascent_rate is not None:
    velocity = ascent_rate

# Load live weather once; without allow_offline_fallback a failure stops the run
offline_fallback_used = False
reduced_levels_used = False
gfs_run_used = None
forecast_hours_used = set()
try:
    load_weather(launch_time=launch_time, latitude=latitude, longitude=longitude,
                 allow_reduced_levels=allow_reduced_levels)
    weather = get_weather(latitude, longitude, altitude, valid_time=launch_time)
    wind_east = weather["wind_u_m_s"]
    wind_north = weather["wind_v_m_s"]
    temperature = weather["temperature_k"]
    pressure = weather["pressure_pa"]
    density = weather["density_kg_m3"]
    reduced_levels_used = weather["reduced_pressure_levels"]
    gfs_run_used = weather["forecast_initialization_utc"]
    use_live_weather = True
    print(f"Live weather loaded: GFS {weather['forecast_initialization_utc']} +{weather['forecast_hour']}h")
except Exception as error:
    if not allow_offline_fallback:
        raise RuntimeError(
            f"Live GFS weather could not be loaded: {error}\n"
            "Set allow_offline_fallback = True to run on the standard atmosphere and constant wind "
            "instead; the result is then not a forecast."
        ) from error
    use_live_weather = False
    offline_fallback_used = True
    temperature, pressure, density = atmosphere(altitude)
    print(f"WARNING: live weather unavailable ({error}); using standard atmosphere and constant wind.")

# The balloon must be able to lift off: buoyancy at launch has to exceed its weight
launch_volume = hydrogen_mass * R_hydrogen * temperature / pressure
free_lift = density * launch_volume - total_mass
if free_lift <= 0:
    raise ValueError(
        f"Balloon cannot lift off: free lift is {free_lift:.3f} kg. At launch it displaces "
        f"{density * launch_volume:.3f} kg of air but has a total mass of {total_mass:.3f} kg "
        f"(payload {payload_mass} kg + balloon {balloon_mass} kg + hydrogen {hydrogen_mass:.3f} kg)."
    )

# Balloon starts out moving with the surface wind
velocity_east = wind_east
velocity_north = wind_north
next_weather_time = 0.0
next_print_time = 0.0
burst = False
burst_point = None
end_reason = None
# Steps whose weather was extrapolated beyond the loaded GFS levels, and the altitudes involved
extrapolated_steps = {"below": 0, "above": 0}
extrapolated_altitudes = {"below": [math.inf, -math.inf], "above": [math.inf, -math.inf]}

path = [(latitude, longitude)]
records = []
while altitude < max_altitude and time < max_time:
    if use_live_weather and time >= next_weather_time:
        try:
            weather = get_weather(latitude, longitude, altitude, valid_time=launch_time + timedelta(seconds=time))
            temperature = weather["temperature_k"]
            pressure = weather["pressure_pa"]
            density = weather["density_kg_m3"]
            relative_humidity = weather["relative_humidity_percent"]
            wind_east = weather["wind_u_m_s"]
            wind_north = weather["wind_v_m_s"]
            forecast_hours_used.update((weather["forecast_hour"], weather["forecast_hour_upper"]))
        except Exception as error:
            if not allow_offline_fallback:
                raise RuntimeError(
                    f"Live weather lookup failed at t={time:.1f}s, altitude {altitude:.0f} m, "
                    f"({latitude:.4f}, {longitude:.4f}): {error}"
                ) from error
            use_live_weather = False
            offline_fallback_used = True
            print(f"WARNING: live weather lookup failed ({error}); switching to standard atmosphere.")
        next_weather_time += weather_update_interval
    if not use_live_weather:
        temperature, pressure,density = atmosphere(altitude)
    gravity = gravity_at(altitude)

    # Whether the weather in use (from the last lookup) was extrapolated beyond the GFS levels
    weather_extrapolated = use_live_weather and weather["extrapolated"]
    if weather_extrapolated:
        side = weather["extrapolation"]
        extrapolated_steps[side] += 1
        extrapolated_altitudes[side][0] = min(extrapolated_altitudes[side][0], altitude)
        extrapolated_altitudes[side][1] = max(extrapolated_altitudes[side][1], altitude)

    if not burst:
        volume = hydrogen_mass * R_hydrogen * temperature / pressure
        if burst_enabled and volume >= max_volume:
            burst = True
            burst_point = (latitude, longitude, altitude, time)
            total_mass = payload_mass + balloon_mass  # hydrogen escapes, payload descends under the parachute
            print(f"Balloon burst at {altitude:.1f}m after {time:.1f}s")
        else:
            volume = min(volume, max_volume)

    # Balloon velocity relative to the surrounding air
    relative_east = velocity_east - wind_east
    relative_north = velocity_north - wind_north
    relative_speed = math.sqrt(relative_east**2 + relative_north**2 + velocity**2)

    if burst:
        volume = 0.0
        area = math.pi * (parachute_diameter / 2.0) ** 2
        current_drag_coefficient = parachute_drag_coefficient
    else:
        radius = (3.0 * volume / (4.0 * math.pi)) ** (1.0 / 3.0)
        area = math.pi * radius**2
        reynolds = density * relative_speed * 2.0 * radius / air_viscosity(temperature)
        current_drag_coefficient = balloon_drag_coefficient(reynolds)

    effective_mass = total_mass + added_mass_coefficient * density * volume
    bouyancy = density * volume * gravity
    drag_factor = 0.5 * density * current_drag_coefficient * area * relative_speed
    drag = drag_factor * velocity
    netForce = bouyancy - (total_mass * gravity) - drag
    acceleration = netForce / effective_mass
    if ascent_rate is not None and not burst and volume < max_volume:
        # Constant-rate ascent: hold the climb at ascent_rate until burst
        acceleration = (ascent_rate - velocity) / dt
    velocity += acceleration * dt
    velocity_east += (-drag_factor * relative_east / effective_mass) * dt
    velocity_north += (-drag_factor * relative_north / effective_mass) * dt
    altitude = max(ground_altitude, altitude + velocity * dt)

    # Position update on the WGS84 ellipsoid
    sin_latitude = math.sin(math.radians(latitude))
    ellipsoid_factor = math.sqrt(1.0 - wgs84_e2 * sin_latitude**2)
    meridian_radius = wgs84_a * (1.0 - wgs84_e2) / ellipsoid_factor**3
    prime_vertical_radius = wgs84_a / ellipsoid_factor
    latitude += math.degrees(velocity_north * dt / (meridian_radius + altitude))
    longitude += math.degrees(velocity_east * dt / ((prime_vertical_radius + altitude) * math.cos(math.radians(latitude))))
    longitude = (longitude + 180.0) % 360.0 - 180.0   # keep within -180..180

    # Icing risk when air below freezing is saturated with respect to ice
    frost_point_k = frost_point(temperature, relative_humidity)
    icing_risk = temperature < 273.15 and temperature <= frost_point_k

    time += dt
    path.append((latitude, longitude))
    records.append({
        "time": time,
        "altitude": altitude,
        "velocity": velocity,
        "acceleration": acceleration,
        "latitude": latitude,
        "longitude": longitude,
        "velocity_east": velocity_east,
        "velocity_north": velocity_north,
        "wind_east": wind_east,
        "wind_north": wind_north,
        "temperature": temperature,
        "pressure": pressure,
        "density": density,
        "relative_humidity": relative_humidity,
        "drag_coefficient": current_drag_coefficient,
        "frost_point": frost_point_k,
        "icing_risk": icing_risk,
        "burst": burst,
        "weather_extrapolated": weather_extrapolated
    })
    if time >= next_print_time:
        print(
            f"Time: {time:.1f}s | "
            f"Altitude: {altitude:.1f}m | "
            f"Velocity: {velocity:.2f}m/s | "
            f"Acceleration: {acceleration}m/s^2 | "
            f"Latitude: {latitude:.5f} | "
            f"Longitude: {longitude:.5f}")
        next_print_time += print_interval

    if burst and altitude <= ground_altitude:
        end_reason = "landed"
        print(f"Landed after {time:.1f}s")
        break

# Why the run stopped; only "landed" means the final point is a landing site
if end_reason is None:
    end_reason = "altitude ceiling" if altitude >= max_altitude else "time limit"
end_reason_detail = {
    "landed": f"landed at {ground_altitude:.0f} m",
    "altitude ceiling": f"altitude ceiling (max_altitude = {max_altitude} m) reached before landing",
    "time limit": f"time limit (max_time = {max_time} s) reached before landing",
}[end_reason]
print(f"Run ended: {end_reason_detail}")
final_point_note = "Landed" if end_reason == "landed" else f"NOT a landing: run stopped at the {end_reason}"
if offline_fallback_used:
    final_point_note += "\nOFFLINE FALLBACK: standard atmosphere and constant wind, not a GFS forecast"

# Create a map centered at the initial position
map_center = path[0]
flight_map = folium.Map(location=map_center, zoom_start=12, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri",
    name="Esri Satellite"
).add_to(flight_map)
folium.PolyLine(path, color="red", weight=4, tooltip="Balloon Trajectory").add_to(flight_map)
folium.Marker(path[0], tooltip="Launch Point", icon=folium.Icon(color="green", icon="play")).add_to(flight_map)
if burst_point is not None:
    folium.Marker(burst_point[:2], tooltip="Burst Point", icon=folium.Icon(color="orange", icon="star"),popup=f"Burst Altitude: {burst_point[2]:.2f}m, \nTime elapsed: {burst_point[3]:.1f}s").add_to(flight_map)
folium.Marker(path[-1], tooltip="Final Point", icon=folium.Icon(color="red", icon="flag"),popup=f"Final Altitude: {altitude:.2f}m, \nTime elapsed: {time:.1f}s" + f"\n{final_point_note}").add_to(flight_map)
flight_map.fit_bounds(path)
if offline_fallback_used:
    flight_map.get_root().html.add_child(folium.Element(
        '<div style="position: fixed; top: 10px; left: 60px; z-index: 1000; padding: 8px 12px; '
        'background: #b00020; color: white; font: bold 14px sans-serif;">'
        'OFFLINE FALLBACK: standard atmosphere and constant wind, not a GFS forecast</div>'
    ))
flight_map.save("balloon_trajectory.html")

print("Map saved as balloon_trajectory.html")

# Warnings about how far the result can be trusted
simulation_warnings = []
if offline_fallback_used:
    simulation_warnings.append("OFFLINE FALLBACK USED: standard atmosphere and constant wind; this is NOT a forecast")
if reduced_levels_used:
    simulation_warnings.append("GFS extended levels were unavailable; weather above about 16 km was extrapolated")
for side, label in [("below", "below the lowest"), ("above", "above the highest")]:
    if extrapolated_steps[side]:
        lowest, highest = extrapolated_altitudes[side]
        simulation_warnings.append(
            f"{extrapolated_steps[side]} of {len(records)} steps "
            f"({100 * extrapolated_steps[side] / len(records):.1f}%) used weather extrapolated "
            f"{label} GFS pressure level (altitudes {lowest:.0f}-{highest:.0f} m)"
        )
if end_reason != "landed":
    simulation_warnings.append(f"the run did not land ({end_reason}); the final point is not a landing site")

# Flight summary
launch_utc = launch_time.astimezone(timezone.utc)
print("\n===== Flight summary =====")
if gfs_run_used is not None:
    print(f"GFS run: {gfs_run_used} | forecast hours used: "
          + ", ".join(f"+{hour}h" for hour in sorted(forecast_hours_used)))
else:
    print("GFS run: none (offline fallback: standard atmosphere and constant wind)")
print(f"Launch time: {launch_utc:%Y-%m-%d %H:%M:%S} UTC")
if burst_point is not None:
    burst_utc = launch_utc + timedelta(seconds=burst_point[3])
    print(f"Burst: {burst_point[2]:.0f} m at {burst_point[3]:.0f} s ({burst_utc:%H:%M:%S} UTC), "
          f"({burst_point[0]:.5f}, {burst_point[1]:.5f})")
else:
    print("Burst: none")
end_utc = launch_utc + timedelta(seconds=time)
if end_reason == "landed":
    print(f"Landing: ({latitude:.5f}, {longitude:.5f}) at {time:.0f} s ({end_utc:%Y-%m-%d %H:%M:%S} UTC)")
else:
    print(f"Landing: none; final point ({latitude:.5f}, {longitude:.5f}) at {altitude:.0f} m "
          f"after {time:.0f} s ({end_utc:%Y-%m-%d %H:%M:%S} UTC)")
max_ascent_rate = max((record["velocity"] for record in records), default=velocity)
print(f"Maximum ascent rate: {max_ascent_rate:.2f} m/s")
print(f"Run ended: {end_reason_detail}")
if simulation_warnings:
    print("Warnings:")
    for warning in simulation_warnings:
        print(f"  WARNING: {warning}")
else:
    print("Warnings: none")

Loading GFS forecast 2026-11-16 18Z +8h...
✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Nov-16 18:00 UTC F08 ┊ GRIB2 @ aws ┊ IDX @ local
GFS forecast loaded successfully.
Loading GFS forecast 2026-11-16 18Z +9h...
✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Nov-16 18:00 UTC F09 ┊ GRIB2 @ aws ┊ IDX @ local
GFS forecast loaded successfully.
Live weather loaded: GFS 2026-11-16T18:00:00Z +8h
Time: 0.1s | Altitude: 0.5m | Velocity: 5.00m/s | Acceleration: 0.0m/s^2 | Latitude: 40.80740 | Longitude: -73.96300
Time: 10.1s | Altitude: 50.5m | Velocity: 5.00m/s | Acceleration: 0.0m/s^2 | Latitude: 40.80693 | Longitude: -73.96277
Time: 20.0s | Altitude: 100.0m | Velocity: 5.00m/s | Acceleration: 0.0m/s^2 | Latitude: 40.80647 | Longitude: -73.96255
Time: 30.0s | Altitude: 150.0m | Velocity: 5.00m/s | Acceleration: 0.0m/s^2 | Latitude: 40.80600 | Longitude: -73.96233
Time: 40.0s | Altitude: 200.0m | Velocity: 5.00m/s | Acceleration: 0.0m/s^2 | Latitude: 40.80553 | Longitude: -73.96210
Time: 